# CEG2003 reproducible project walkthrough

This notebook explains the executed project. It reads saved results rather than repeatedly importing four million Excel rows. Rebuild with `run_project.py` when source data or settings change. The main experiment stays within March; April is supplementary. Read the report for assumptions and limits.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'project_config.json').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from ceg2003.features import FEATURES, build_features, label_features
config = json.loads((ROOT / 'project_config.json').read_text())
print('Project root:', ROOT)
print('Features:', FEATURES)

## 1. Source and cleaning controls

Raw event counts, exact-key repeats and unassigned events reconcile independently. Source hashes identify the exact inputs. Missing identities remain visible in fleet totals.

In [ ]:
audit = json.loads((ROOT / 'results/quality_audit.json').read_text())
print(pd.read_csv(ROOT / 'results/cleaning_reconciliation.csv').to_string(index=False))
print('Duration mismatches:', audit['raw']['duration_mismatches'])
print('Incident policy:', audit['incidents_policy'])

## 2. Feature construction

Rates divide by alarm-bearing days, not complete driving exposure. The window is selected before labels are attached. The final-test feature window ends before its outcome window.

In [ ]:
daily = pd.read_parquet(ROOT / 'data/processed/captain_daily.parquet')
accidents = pd.read_parquet(ROOT / 'data/processed/accidents_audit.parquet')
window = config['windows'][2]
features = build_features(daily, window, config['minimum_observed_days'])
labelled = label_features(features, accidents, window)
eligible = labelled.loc[labelled.sufficient_history]
print('Window:', window)
print('Eligible captains:', len(eligible), 'Recorded positives:', int(eligible.outcome.sum()))
print(eligible[FEATURES].describe().round(3).to_string())

## 3. Chronological splits and selection

Each row is a captain-window. Captains can recur across time because the task concerns later forecasts for existing captains. Selection uses validation AP; the test cannot choose the winner.

In [ ]:
print(pd.read_csv(ROOT / 'results/primary_march/split_summary.csv').to_string(index=False))
print(pd.read_csv(ROOT / 'results/primary_march/validation_metrics.csv')[['model','average_precision']].to_string(index=False))
summary = json.loads((ROOT / 'results/project_summary.json').read_text())
print('Selected:', summary['primary']['selected_model'])

## 4. Frozen test results

Precision/recall at K use a fixed 10% review capacity. A positive outcome means recorded involvement under the provisional policy, not fault. The very small positive sample and weak overall discrimination limit the result.

In [ ]:
metrics = pd.read_csv(ROOT / 'results/primary_march/test_metrics.csv')
print(metrics[['model','average_precision','roc_auc','reviewed','precision_at_k','recall_at_k']].round(4).to_string(index=False))
print('Uncertainty:', summary['primary']['test_uncertainty'])

## 5. Verify the review arithmetic

The selected test list contains two of seven recorded positives. Compute the quantities from the saved predictions, independently of the metrics table.

In [ ]:
pred = pd.read_csv(ROOT / 'results/primary_march/test_predictions.csv')
selected = pred.loc[pred.model == summary['primary']['selected_model']]
reviewed = selected.loc[selected.selected_for_review]
tp = int(reviewed.outcome.sum())
print('Reviewed:', len(reviewed), 'Captured:', tp)
print('Precision:', tp / len(reviewed))
print('Recall:', tp / int(selected.outcome.sum()))

## 6. Demonstration versus evaluation

The separate demonstration model refits after evaluation and scores 25-31 March profiles. April outcomes do not enter that ranking. Low-history captains retain missing scores.

In [ ]:
review = pd.read_parquet(ROOT / 'data/processed/captain_review.parquet')
print('Sufficient history:', int(review.sufficient_history.sum()))
print('Insufficient history:', int((~review.sufficient_history).sum()))
assert review.loc[~review.sufficient_history, 'score'].isna().all()
print('Evaluation and demonstration models are separately saved in models/.')

## 7. Supplement and remaining requirements

April results are a single-cohort captain holdout with unconfirmed scope and completeness. The actual face-on-camera video, official peer assessment, group details and LMS submission remain human tasks.

In [ ]:
print('Supplement:', summary['supplemental']['validation_design'])
print(pd.read_csv(ROOT / 'results/requirements_checklist.csv')[['requirement','status']].to_string(index=False))

## Rebuild and run

From the project root: `.venv\Scripts\python.exe run_project.py`, then `python -m pytest`, then the report/support/package builders. Launch the dashboard with `launch_dashboard.ps1`. See `docs/BUILD_PROCESS.md` for the full sequence and every assumption.